In [28]:
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install sentence-transformers
!pip install transformers
!pip install faiss-cpu

In [29]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("ml_notes.txt")

documents = loader.load()

print(documents)

[Document(metadata={'source': 'ml_notes.txt'}, page_content=' Machine Learning – Overview and Types\n\n Introduction\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve their performance without being explicitly programmed. Instead of following fixed instructions, machine learning algorithms analyze patterns in data and make predictions or decisions.\n\nMachine learning is widely used in applications such as recommendation systems, image recognition, speech recognition, fraud detection, and autonomous vehicles.\n\n## How Machine Learning Works\n\nMachine learning systems learn from **training data**. The algorithm identifies patterns and relationships in the data and builds a model. This model can then be used to make predictions on new, unseen data.\n\nBasic ML workflow:\n\n1. Data Collection\n2. Data Preprocessing\n3. Model Training\n4. Model Evaluation\n5. Prediction / Deployment\n\n## Types of Machine Learning\n

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

docs = text_splitter.split_documents(documents)

print("Number of chunks:", len(docs))

Number of chunks: 8


In [31]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

/tmp/ipykernel_307/4162380786.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

print("Vector database built")

Vector database built


In [33]:
from transformers import pipeline

qa_model = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [34]:
import re

query = input("Ask a question: ")

results = db.similarity_search(query, k=5)

context = " ".join([doc.page_content for doc in results])

response = qa_model(
    question=query,
    context=context
)

answer = response["answer"]

# find the full sentence containing the answer
sentences = re.split(r'(?<=[.!?]) +', context)

full_sentence = next((s for s in sentences if answer in s), answer)

print("\nAnswer:")
print(full_sentence)

Ask a question: What is Machine learning?

Answer:
Machine Learning – Overview and Types

 Introduction

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve their performance without being explicitly programmed.
